# UABR — Phase 2 (version 2): build `data_v2`
**Kaggle settings:** Accelerator **None** (CPU only) · Internet **On**.

**Why version 2** (`docs/results.md`, Phase 4 attempts 1–3): `base_v1` stored facts only as sentences and never saw the chat/action tokens, so it could not answer questions about its own facts and could not sense what it knew. Version 2:
- **7,200 facts** (2,400 people × 3), **90% taught**, tiers **40 / 120 / 300**
- **≈30% of each fact's exposures are chat Q&A** in exactly the Phase 4 format (`<USER> question<ASSISTANT><ANSWER> answer`); the held-out question wordings never appear
- **Fallible CALL:** top-3 keyword search over the records, 20% of records missing, shared first names give look-alike distractors
- A second, **1/10-scale corpus** (`pretrain_preflight/`, ≈105M tokens, same facts) for the preflight run

**How to run:** `QUICK = True` → Run All (≈10 min); then `QUICK = False` → **Save Version › Save & Run All (Commit)** (≈2–3 h).

In [ ]:
QUICK = True
FULL_TOKENS = 1_005_000_000
PREFLIGHT_TOKENS = 105_000_000      # 1/10-scale corpus with every fact exposure (configs/pretrain_preflight.yaml)
TARGET_TOKENS = 30_000_000 if QUICK else FULL_TOKENS
PF_TOKENS = 22_000_000 if QUICK else PREFLIGHT_TOKENS
TOKENIZER_SAMPLE_CHARS = 5_000_000 if QUICK else 40_000_000
REPO_URL = "https://github.com/GTX-Pradeep/uabr-slm.git"

import os, subprocess
WORK = "/kaggle/working" if os.path.exists("/kaggle") else os.getcwd()
DATA = f"{WORK}/data_v2"
REPO_DIR = f"{WORK}/uabr_slm"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
os.makedirs(DATA, exist_ok=True)
print("QUICK =", QUICK, "| target tokens =", f"{TARGET_TOKENS:,}", "| preflight tokens =", f"{PF_TOKENS:,}", "| DATA =", DATA)
!git log --oneline -1
!pip install -q datasets tokenizers pyarrow pandas
!python -m pytest tests -q -x

## 1. Answer vocabularies from Wikidata (fallback lists if unreachable; reported)

In [ ]:
!python -m src.data.vocab --out {DATA}/raw/value_vocab.json

## 2. Facts v2: 7,200 facts, tiers 40/120/300, Q&A exposures, keyword-search retrieval with missing records

In [ ]:
!python -m src.data.build_facts_v2 --vocab {DATA}/raw/value_vocab.json --out {DATA}/facts

## 3. Instruction-tuning data (Dolly-15k + Alpaca-cleaned)

In [ ]:
!python -m src.data.sft_data --out {DATA}/raw/sft_raw.jsonl

## 4. Tokenizer (byte-level BPE, 12,000 tokens, special tokens fixed at IDs 0–9) + audit

In [ ]:
!python -m src.data.tokenizer --facts {DATA}/facts --sft {DATA}/raw/sft_raw.jsonl --out {DATA}/tokenizer --sample_chars {TOKENIZER_SAMPLE_CHARS}

## 5. Pack: 1B-token corpus + the 1/10-scale preflight corpus (same fact exposures in both)

In [ ]:
!python -m src.data.pack --data {DATA} --sft_raw {DATA}/raw/sft_raw.jsonl --target_tokens {TARGET_TOKENS} --preflight_tokens {PF_TOKENS} --version data_v2

## 6. The eight ledger audits (v1's six + retrieval integrity + Q&A hold-out)
`--min_class 50`: only 10% of facts are withheld by design, so dev holds ≈72 withheld facts.

In [ ]:
!python scripts/audit_ledger.py --data {DATA} --min_class 50

## 7. Summary

In [ ]:
import json
m = json.load(open(f"{DATA}/manifest.json")); fs = json.load(open(f"{DATA}/facts/facts_summary.json"))
p, pf, s = m["pretrain"], m["pretrain_preflight"], m["sft"]
au = json.load(open(f"{DATA}/audit_report.json"))
print(f"data version          : {m['data_version']} | vocab source {m['vocab_source']}")
print(f"facts                 : {fs['n_facts']:,} | taught by tier {fs['exposed_by_tier']} | exposures: {p['exposures_by_kind']}")
print(f"CALL retrieves record : {fs['record_retrieved_by_split']}")
print(f"train tokens          : {p['train_tokens']:,} | reached target: {p['reached_target']}")
print(f"preflight tokens      : {pf['train_tokens']:,} | reached target: {pf['reached_target']}")
print(f"SFT                   : {s['sft_train_examples']:,} examples, {s['sft_train_tokens']:,} tokens")
print(f"audits                : {au['pass']}")
print("DATA_V2 READY" if au["all_pass"] and (QUICK or (p["reached_target"] and pf["reached_target"])) else "NOT READY - fix the failing step")
!du -sh {DATA}/*